# Week 14 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 14 Day 01: Bond cashflows and pricing mechanics

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Develop working fixed-income intuition for pricing, curve dynamics, and interest-rate risk.

## Continuity and Handoff
- Previous checkpoint: Week 13 Day 07: Portfolio Project
- Previous lesson file: content/week-13/day-07.md
- Today's deliverable: Implement a simple bond pricer for coupon bonds.
- Next handoff target: Week 14 Day 02: Yield curve structure
- Next lesson file: content/week-14/day-02.md

## Theory Concepts

### Concept 1: Coupon and principal cashflows
Coupon and principal cashflows is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Yield-to-maturity intuition
Yield-to-maturity intuition is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Price-yield inverse relation
Price-yield inverse relation is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Price a fixed-coupon bond across multiple yield assumptions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Bond cashflows and pricing mechanics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement a simple bond pricer for coupon bonds.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 14 Day 01, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Bond cashflows and pricing mechanics' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Bond cashflows and pricing mechanics as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
How does reinvestment assumption affect interpretation of yield?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 14 Day 01 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [2]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1401)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033826760101,
 'portfolio_volatility': 0.07641256138634328,
 'turnover_proxy': 0.2575701289920008}

## Week 14 Day 01 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [3]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 115.000
price_t = 115.977
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.008496)
print('  gross_return_expected  =', 1.008496)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 115.0
  P_t    : 115.977
  r_t    : 0.008496 => 0.85%
  1+r_t  : 1.008496

Expected numeric check:
  simple_return_expected = 0.008496
  gross_return_expected  = 1.008496

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 14 Day 02: Yield curve structure

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Develop working fixed-income intuition for pricing, curve dynamics, and interest-rate risk.

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 01: Bond cashflows and pricing mechanics
- Previous lesson file: content/week-14/day-01.md
- Today's deliverable: Create yield-curve plotting utility and slope diagnostics.
- Next handoff target: Week 14 Day 03: Duration and convexity
- Next lesson file: content/week-14/day-03.md

## Theory Concepts

### Concept 1: Spot, forward, and par rates
Spot, forward, and par rates is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Curve shapes and macro context
Curve shapes and macro context is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Term premium intuition
Term premium intuition is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Construct a toy yield curve and interpret steepening/flattening moves.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Yield curve structure'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create yield-curve plotting utility and slope diagnostics.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 14 Day 02, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Yield curve structure' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Yield curve structure as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
Which macro regime tends to produce curve inversion?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 14 Day 02 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [4]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1402)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033382932732,
 'portfolio_volatility': 0.07641255429878707,
 'turnover_proxy': 0.2575702002994425}

## Week 14 Day 02 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [5]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 116.000
price_t = 117.044
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.009000)
print('  gross_return_expected  =', 1.009000)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 116.0
  P_t    : 117.044
  r_t    : 0.009 => 0.90%
  1+r_t  : 1.009

Expected numeric check:
  simple_return_expected = 0.009
  gross_return_expected  = 1.009

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 14 Day 03: Duration and convexity

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Develop working fixed-income intuition for pricing, curve dynamics, and interest-rate risk.

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 02: Yield curve structure
- Previous lesson file: content/week-14/day-02.md
- Today's deliverable: Build duration-convexity sensitivity calculator.
- Next handoff target: Week 14 Day 04: Rate-risk scenario analysis
- Next lesson file: content/week-14/day-04.md

## Theory Concepts

### Concept 1: Macaulay and modified duration
Macaulay and modified duration is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: First-order rate sensitivity
First-order rate sensitivity is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Convexity correction
Convexity correction is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Risk Contribution
$$
RC_i=w_i\frac{(\Sigma w)_i}{\sigma_p}
$$
**Plain-English interpretation:** Per-position risk budget.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Estimate bond price change under rate shocks using duration/convexity.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Duration and convexity'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build duration-convexity sensitivity calculator.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 14 Day 03, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Duration and convexity' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Duration and convexity as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
When does duration-only approximation fail materially?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 14 Day 03 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [6]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1403)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894034586904552,
 'portfolio_volatility': 0.07641258745266512,
 'turnover_proxy': 0.25757031344454934}

## Week 14 Day 03 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [7]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 117.000
price_t = 118.112
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.009504)
print('  gross_return_expected  =', 1.009504)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 117.0
  P_t    : 118.112
  r_t    : 0.009504 => 0.95%
  1+r_t  : 1.009504

Expected numeric check:
  simple_return_expected = 0.009504
  gross_return_expected  = 1.009504

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 14 Day 04: Rate-risk scenario analysis

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Develop working fixed-income intuition for pricing, curve dynamics, and interest-rate risk.

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 03: Duration and convexity
- Previous lesson file: content/week-14/day-03.md
- Today's deliverable: Implement scenario engine for curve shocks.
- Next handoff target: Week 14 Day 05: Fixed-income portfolio interpretation
- Next lesson file: content/week-14/day-05.md

## Theory Concepts

### Concept 1: Parallel and non-parallel shifts
Parallel and non-parallel shifts is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Curve-bucket exposures
Curve-bucket exposures is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Stress scenario design
Stress scenario design is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Duration Shock
$$
\frac{\Delta P}{P}\approx-D_{mod}\Delta y
$$
**Plain-English interpretation:** First-order bond sensitivity.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Run parallel and twist scenarios on a small bond basket.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Rate-risk scenario analysis'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Implement scenario engine for curve shocks.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 14 Day 04, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Rate-risk scenario analysis' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Rate-risk scenario analysis as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
Which scenario reveals the largest hidden risk?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 14 Day 04 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [8]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1404)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894034512950564,
 'portfolio_volatility': 0.07641256976271175,
 'turnover_proxy': 0.2575703109298928}

## Week 14 Day 04 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [9]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 118.000
price_t = 119.180
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.010000)
print('  gross_return_expected  =', 1.010000)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 118.0
  P_t    : 119.18
  r_t    : 0.01 => 1.00%
  1+r_t  : 1.01

Expected numeric check:
  simple_return_expected = 0.01
  gross_return_expected  = 1.01

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 14 Day 05: Fixed-income portfolio interpretation

## Study Duration
- Planned effort: 4 hours

## 5-Block Daily Structure
- **Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Block 4 (60 min):** Python/pandas implementation and output verification.
- **Block 5 (30 min):** Practice questions, interview drill, and reflection.

## Why It Matters in Quant
Develop working fixed-income intuition for pricing, curve dynamics, and interest-rate risk.

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 04: Rate-risk scenario analysis
- Previous lesson file: content/week-14/day-04.md
- Today's deliverable: Generate fixed-income risk report with key sensitivities.
- Next handoff target: Week 14 Day 06: Revision Sprint
- Next lesson file: content/week-14/day-06.md

## Theory Concepts

### Concept 1: Carry and roll-down intuition
Carry and roll-down intuition is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Interest-rate hedging basics
Interest-rate hedging basics is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Risk-report communication
Risk-report communication is a core part of 'Fixed income basics for quant workflows'. Start with notation discipline: define weights, constraints, and risk units before solving the allocation problem. Then focus on allocation constraints, risk decomposition, and capital efficiency by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: CVaR
$$
CVaR_\alpha=E[L\mid L\ge VaR_\alpha]
$$
**Plain-English interpretation:** Tail-risk expectation.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 2: Portfolio Return
$$
\mu_p=w^\top\mu
$$
**Plain-English interpretation:** Expected return from weighted assets.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

### Formula 3: Portfolio Variance
$$
\sigma_p^2=w^\top\Sigma w
$$
**Plain-English interpretation:** Quadratic risk engine.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Evaluate the metric on at least three assets and document which constraint changes the final portfolio most.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $w$ | Portfolio weights | sum to 1 | [0.35,0.25,0.40] |
| $\Sigma$ | Covariance matrix | return^2 | 3x3 matrix |
| $D_{mod}$ | Modified duration | years | 5.8 |

## Real Trading Example
- Instruments: SPY, TLT, GLD, HYG
- Macro overlay (FRED): DGS10, T10YIE
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Summarize fixed-income portfolio behavior under changing rates.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Fixed-income portfolio interpretation'.

## Step-by-Step Solved Problems
### Solved Problem 1: Portfolio expected return
Given:
- w=[0.6,0.4], mu=[0.12,0.08].
Solution:
1. $\mu_p=w^\top\mu$.
2. mu_p = 0.6*0.12 + 0.4*0.08 = 0.104.
Final answer: Portfolio expected return = 10.4%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Portfolio volatility
Given:
- sigma1=0.20, sigma2=0.12, rho=0.30, w1=0.6, w2=0.4.
Solution:
1. $\sigma_p^2=w_1^2\sigma_1^2+w_2^2\sigma_2^2+2w_1w_2\rho\sigma_1\sigma_2$.
2. sigma_p^2 = 0.02048.
3. sigma_p = sqrt(0.02048) = 0.1431.
Final answer: Portfolio volatility = 14.31%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: Duration shock
Given:
- Modified duration = 5.8, yield shift = +0.25%.
Solution:
1. $\Delta P/P\approx-D_{mod}\Delta y$.
2. DeltaP/P = -5.8*0.0025 = -0.0145.
Final answer: Approximate bond price change = -1.45%.
Common trap: Ignoring covariance and focusing only on expected return, which underestimates portfolio risk.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Generate fixed-income risk report with key sensitivities.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
mu, cov = estimate_moments(asset_returns)
weights = solve_constrained_mv(mu, cov, max_weight=0.35)
risk_budget = risk_contributions(weights, cov)
rebalance_flag = should_rebalance(weights, target_weights, threshold=0.03)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. In Week 14 Day 05, explain one formula from today's lesson in plain language and define every symbol used.
2. Using one real asset from today's universe, compute the metric and state one risk guardrail you would enforce.
3. Interview drill: In 60 seconds, explain why 'Fixed-income portfolio interpretation' matters for production trading systems.

Answer key template:
- Q1: Formula + symbol table + units.
- Q2: Numeric result + interpretation + guardrail.
- Q3: One concise story linking model, risk, and execution.

### Interview Drill
- Prompt: "Walk me through Fixed-income portfolio interpretation as if you are presenting to a PM who cares about risk-adjusted returns."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision.
  3. Awareness of edge cases, costs, and failure modes.

## Reflection Question
How would you communicate fixed-income risk to an equity-focused PM?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


## Week 14 Day 05 Runnable Example
Run this cell, inspect outputs, then answer the quiz.

In [10]:
# Portfolio/risk demo: multi-asset allocation on real prices
np.random.seed(1405)
tickers = ['SPY', 'TLT', 'GLD', 'HYG']
prices = load_market_prices(tickers, start='2012-01-01')
returns = prices.pct_change().dropna()
mu = returns.mean().values * 252
cov = returns.cov().values * 252
vol = np.sqrt(np.diag(cov))

# inverse-volatility heuristic with normalization
inv_vol = 1 / np.maximum(vol, 1e-8)
weights = inv_vol / inv_vol.sum()
port_return = float(weights @ mu)
port_vol = float(np.sqrt(weights @ cov @ weights))
turnover_proxy = float(np.abs(np.diff(weights)).sum())

{
    'tickers': tickers,
    'weights': [float(round(x, 4)) for x in weights],
    'portfolio_expected_return': port_return,
    'portfolio_volatility': port_vol,
    'turnover_proxy': turnover_proxy,
}


{'tickers': ['SPY', 'TLT', 'GLD', 'HYG'],
 'weights': [0.1887, 0.2164, 0.1938, 0.4011],
 'portfolio_expected_return': 0.06894033689799556,
 'portfolio_volatility': 0.07641260261006134,
 'turnover_proxy': 0.25757032067392094}

## Week 14 Day 05 Quiz

Answer these before revealing the solution cell:
1. Write one formula from today in plain language and define each symbol.
2. Use one real ticker from today's examples and state one risk guardrail.
3. In one sentence: why does this concept matter for live trading decisions?

Then run the next code cell to compare against a reference answer template.

In [11]:
# Quiz reference solution template (auto-generated)
price_t_minus_1 = 119.000
price_t = 120.249
r_t = (price_t - price_t_minus_1) / price_t_minus_1
gross = 1 + r_t

print('Reference symbols:')
print('  P_(t-1):', price_t_minus_1)
print('  P_t    :', price_t)
print('  r_t    :', round(r_t, 6), '=>', f'{r_t*100:.2f}%')
print('  1+r_t  :', round(gross, 6))

print('\nExpected numeric check:')
print('  simple_return_expected =', 0.010496)
print('  gross_return_expected  =', 1.010496)

print('\nInterview-style answer template:')
print('  Formula in words: return = price change / starting price')
print('  Risk guardrail: reject signals when volatility regime shifts above threshold')


Reference symbols:
  P_(t-1): 119.0
  P_t    : 120.249
  r_t    : 0.010496 => 1.05%
  1+r_t  : 1.010496

Expected numeric check:
  simple_return_expected = 0.010496
  gross_return_expected  = 1.010496

Interview-style answer template:
  Formula in words: return = price change / starting price
  Risk guardrail: reject signals when volatility regime shifts above threshold


# Week 14 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 05: Fixed-income portfolio interpretation
- Previous lesson file: content/week-14/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 14 Day 07: Portfolio Project
- Next lesson file: content/week-14/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Recompute duration/convexity for one bond from scratch
- Review curve scenario assumptions
- Summarize fixed-income glossary in one page

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 14 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 14 Day 06: Revision Sprint
- Previous lesson file: content/week-14/day-06.md
- Today's deliverable: Bond sensitivity analysis
- Next handoff target: Week 15 Day 01: Option payoff structure
- Next lesson file: content/week-15/day-01.md

## Project Title
Bond sensitivity analysis

## Problem Statement
Build a fixed-income sensitivity dashboard for a small bond portfolio.

## Data Sources
- Synthetic bond set
- Yield curve snapshots
- Rate shock scenarios

## Implementation Steps
1. Price bond set
2. Compute duration and convexity
3. Run rate scenarios
4. Aggregate portfolio risk
5. Document key exposures

## Evaluation Metrics
- DV01 proxy
- Convexity impact
- Scenario loss
- Hedge suggestion quality

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
